In [1]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

c:\Users\arekr\OneDrive\Pulpit\Semestr6\SSNE\ssne_26L\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("Data/hate_train.csv")

print(df.head())
print(df.columns)
print(df.shape)

print("\nRozkład klas:")
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True))

                                            sentence  label
0  Dla mnie faworytem do tytułu będzie Cracovia. ...      0
1  @anonymized_account @anonymized_account Brawo ...      0
2  @anonymized_account @anonymized_account Super,...      0
3  @anonymized_account @anonymized_account Musi. ...      0
4    Odrzut natychmiastowy, kwaśna mina, mam problem      0
Index(['sentence', 'label'], dtype='str')
(10041, 2)

Rozkład klas:
label
0    9190
1     851
Name: count, dtype: int64
label
0    0.915247
1    0.084753
Name: proportion, dtype: float64


In [3]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))

Train: 8032
Validation: 2009


In [4]:
MODEL_NAME = "allegro/herbert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [5]:
def tokenize(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [6]:
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")

train_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map: 100%|██████████| 2009/2009 [00:00<00:00, 7544.90 examples/s]


In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights: 100%|██████████| 199/199 [00:00<?, ?it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; 

In [9]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds)
    }

In [11]:
training_args = TrainingArguments(
    output_dir="./hate_model",

    num_train_epochs=8,

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    logging_steps=50,

    fp16=torch.cuda.is_available()
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

In [13]:
trainer.train()

c:\Users\arekr\OneDrive\Pulpit\Semestr6\SSNE\ssne_26L\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
results = trainer.evaluate()

print(results)

In [ ]:
preds = trainer.predict(val_ds)

y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

print(
    classification_report(
        y_true,
        y_pred,
        digits=4
    )
)

In [ ]:
cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm
)

disp.plot(cmap="Blues")

plt.title("Confusion Matrix")
plt.show()

In [ ]:
trainer.save_model("./best_hate_model")

tokenizer.save_pretrained(
    "./best_hate_model"
)